# CT + MRI Combined End-to-End Workflow

This notebook sends CT and MRI images through the same complete prototype workflow: ingestion, modality-aware preprocessing, feature extraction, triage prediction, explainability, and combined study reporting.

In [ ]:
from pathlib import Path

from config import DATA_DIR
from src.dataset_loader import build_dataset_rows
from src.triage_pipeline import explain_prediction, triage_decision

print(f'Dataset root: {DATA_DIR}')
print('Combined CT + MRI workflow is ready.')

In [ ]:
def collect_modality_images(ct_paths=None, mri_paths=None):
    """Build one mixed study list while preserving the source modality."""
    study_items = [(Path(path), 'CT') for path in (ct_paths or [])]
    study_items.extend((Path(path), 'MRI') for path in (mri_paths or []))
    return study_items


dataset_rows = build_dataset_rows(DATA_DIR)
print(f'Ingested labeled image records: {len(dataset_rows)}')
print('The application accepts separate CT and MRI uploads in one combined study, maximum 40 images total.')

In [ ]:
def analyze_combined_study(ct_paths, mri_paths, max_images=40):
    study_items = [(Path(path), 'CT') for path in ct_paths]
    study_items.extend((Path(path), 'MRI') for path in mri_paths)
    if len(study_items) > max_images:
        raise ValueError(f'A combined study cannot contain more than {max_images} images.')

    results = []
    for image_path, modality in study_items:
        result = triage_decision(image_path, modality)
        ranked_features = explain_prediction(result)
        results.append({
            'image': image_path.name,
            'modality': result.modality,
            'prediction': result.predicted_label,
            'priority': result.priority,
            'confidence': result.confidence,
            'features': result.features,
            'top_features': ranked_features,
        })
    return results

In [ ]:
# Use the first available labeled images as a reproducible example.
ct_paths = [row['path'] for row in dataset_rows[:20]]
mri_paths = [row['path'] for row in dataset_rows[20:40]]

if ct_paths or mri_paths:
    combined_results = analyze_combined_study(ct_paths, mri_paths)
    for item in combined_results:
        print(f"{item['modality']} | {item['image']} | {item['prediction']} | {item['priority']} | {item['confidence']:.2%}")
else:
    combined_results = []
    print('No images found. Add labeled images under data/train, data/val, or data/test.')

In [ ]:
def summarize_combined_results(results):
    summary = {}
    for item in results:
        modality = item['modality']
        summary.setdefault(modality, {'images': 0, 'urgent_or_critical': 0})
        summary[modality]['images'] += 1
        if item['priority'] in {'Urgent', 'Critical'}:
            summary[modality]['urgent_or_critical'] += 1
    return summary

print('Combined study summary:')
print(summarize_combined_results(combined_results))